# Evidence-per-Token: Tier A reproduction

This notebook reproduces every number and figure in the paper *When does code-graph structure help a token-budgeted coding agent? A matched-budget study of context selection on the Gemma 4 Developer Agent graphs*.

**What it does.** It evaluates ten context selectors (BM25, dense, hybrid, graph expansion, connected, connected + multi-level packing, random and oracle controls) on all 129 public tasks at budgets of 2k, 4k, 8k and 16k estimated tokens. It reports function-level recall of the code edited by the reference fix. Every configuration is tuned by 5-fold task-level cross-validation, and every reported number is out-of-fold.

**Inputs.** Attach the competition dataset (*Google - The Gemma 4 Developer Agent Competition*) as an input. The notebook reads only `tasks.jsonl`, `graphs/` and `embeddings/`, and copies no competition data into its outputs except derived metrics.

**Leakage boundary.** Selectors read only `problem_statement`, the graphs and the embeddings. `patch` is used only for gold labels, the oracle and metrics. `test_patch` is never read.

**Runtime.** CPU only, no GPU or internet needed. About 45 to 55 minutes on a 4-core Kaggle CPU session: grid about 4 min, diagnostics about 4 min, robustness extensions about 6 min, RL (three seeds) about 30 min.

Code licence: Apache-2.0.

In [ ]:
import glob, os, subprocess, sys

cands = glob.glob("/kaggle/input/**/tasks.jsonl", recursive=True)
DATA = os.path.dirname(cands[0]) if cands else os.environ.get("EPT_DATA", "data")
assert os.path.exists(os.path.join(DATA, "tasks.jsonl")), f"tasks.jsonl not found under {DATA}; attach the competition dataset"
assert os.path.isdir(os.path.join(DATA, "graphs")) and os.path.isdir(os.path.join(DATA, "embeddings"))
WORKERS = max(1, (os.cpu_count() or 2))
print("data:", DATA, "| workers:", WORKERS)
os.makedirs("evtok", exist_ok=True)
os.makedirs("tests", exist_ok=True)

def run(*args):
    # Run an evtok module and stream its output live, so long steps show progress.
    import time
    t0 = time.time()
    print("running:", " ".join(args), flush=True)
    p = subprocess.Popen([sys.executable, "-u", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"[exit code {rc}, {time.time() - t0:.0f}s]", flush=True)
    if rc != 0:
        raise RuntimeError(f"step failed with exit code {rc}; see the output above")

## 1. Source code

Each cell below writes one module of the `evtok` package. The same files are in the paper's code release.

| Module | Role |
|---|---|
| `textproc` | identifier-aware tokenization, token-cost estimate, signature extraction |
| `data` | load tasks, graphs (node-link JSON), embeddings (`.npz` keyed by node id) |
| `gold` | map the reference patch to graph nodes (**evaluation only**) |
| `retrieval` | BM25, identifier anchors, dense pseudo-query, hybrid score (problem statement only) |
| `selectors` | all selectors; each guarantees packet cost ≤ budget |
| `metrics` | Recall@B, Hit@B, any-tier recall, file hit, components |
| `run_tier_a` | evaluate every configuration on every task |
| `analyze` | 5-fold CV selection, paired bootstrap, McNemar, tables |
| `diagnostics` | graph-distance headroom analysis |
| `figures` | recall-vs-budget figure |
| `extensions` | learned ranker, budget-split rescue, Steiner-connected top-k (Section 6.3) |
| `export_labels` | regenerate localization labels (evaluation set) |
| `rl_select` | REINFORCE context selection with token penalties (Section 6.4) |

In [ ]:
%%writefile evtok/__init__.py
"""Evidence-per-Token: budgeted, connected, multi-resolution context selection over code graphs.

Licensed under Apache-2.0.
"""

In [ ]:
%%writefile evtok/textproc.py
"""Text utilities: identifier-aware tokenization, query cleaning, token-cost estimates, tier rendering."""
from __future__ import annotations

import re

# Token cost estimate. The Gemma tokenizer is not available offline on the dev machine;
# characters-per-token is calibrated later against the real tokenizer (see README).
CHARS_PER_TOKEN = 3.6

_WORD = re.compile(r"[A-Za-z_][A-Za-z0-9_]*|\d+")
_CAMEL = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+")

STOP = set(
    """a an the and or of to in on for with by is are be was were this that it its as at from
    if else elif not no but we you they he she i do does did can could should would will may
    have has had into than then there here which who what when where how why all any some
    self cls none true false return def class import pass raise none ok use using used new
    pr pull request description discussion please link fix fixes add adds added https http www com github
    """.split()
)


def ident_tokens(s: str) -> list[str]:
    """Split text into lowercase tokens; identifiers are also split on snake_case and camelCase.

    The full identifier is kept as an extra token so exact-name matches score higher.
    """
    out: list[str] = []
    for w in _WORD.findall(s):
        lw = w.lower()
        parts = [p.lower() for part in w.split("_") if part for p in _CAMEL.findall(part)]
        if len(parts) > 1 and lw not in STOP:
            out.append(lw)
        out.extend(p for p in parts if p not in STOP and len(p) > 1)
        if len(parts) <= 1 and lw not in STOP and len(lw) > 1 and lw not in parts:
            out.append(lw)
    return out


_HTML_COMMENT = re.compile(r"<!--.*?-->", re.S)
_URL = re.compile(r"https?://\S+")


def clean_problem(ps: str) -> str:
    """Remove PR-template comments and URLs from a problem statement."""
    ps = _HTML_COMMENT.sub(" ", ps)
    ps = _URL.sub(" ", ps)
    return ps


def est_tokens(s: str) -> int:
    return int(len(s) / CHARS_PER_TOKEN) + 1


def signature_text(text: str) -> str:
    """Return the definition header plus the first docstring paragraph (if any)."""
    lines = text.split("\n")
    out: list[str] = []
    i = 0
    # header: decorators + def/class line(s) until a line ending with ':'
    seen_def = False
    while i < len(lines):
        out.append(lines[i])
        seen_def = seen_def or re.match(r"\s*(async\s+def|def|class)\b", lines[i]) is not None
        if seen_def and lines[i].rstrip().endswith(":"):
            i += 1
            break
        i += 1
        if i > 20:
            break
    # docstring
    while i < len(lines) and not lines[i].strip():
        i += 1
    if i < len(lines) and lines[i].strip()[:3] in ('"""', "'''"):
        q = lines[i].strip()[:3]
        doc = [lines[i]]
        if not (lines[i].strip().count(q) >= 2 and len(lines[i].strip()) > 3):
            i += 1
            while i < len(lines) and len(doc) < 8:
                doc.append(lines[i])
                if q in lines[i] or not lines[i].strip():
                    break
                i += 1
        out.extend(doc)
    return "\n".join(out)


def norm_line(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
%%writefile evtok/data.py
"""Loading tasks, code graphs (node-link JSON) and node embeddings (.npz keyed by node id)."""
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field

import numpy as np

from .textproc import est_tokens, signature_text

TIERS = ("outline", "sig", "full")


def load_tasks(data_dir: str) -> list[dict]:
    with open(os.path.join(data_dir, "tasks.jsonl"), encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def graph_key(task: dict) -> str:
    return task["repo"].split("/")[1] + "_" + task["base_commit"]


def is_test_node(node_id: str) -> bool:
    parts = node_id.split(".")
    return any(p in ("tests", "test", "testing", "conftest") or p.startswith("test_") for p in parts)


@dataclass
class RepoGraph:
    key: str
    ids: list[str]
    text: list[str]
    idx: dict[str, int]
    nbrs: list[np.ndarray]  # undirected neighbours (dedup, no self loops)
    emb: np.ndarray  # (n, d), L2-normalized; zero rows for missing vectors
    has_emb: np.ndarray  # bool (n,)
    cost: np.ndarray  # (n, 3) token cost per tier: outline, sig, full
    pool: np.ndarray  # bool (n,): eligible for context (non-test)
    extra: dict = field(default_factory=dict)
    succ: list = field(default_factory=list)  # directed `calls` successors (used only for packet rendering)


def load_graph(data_dir: str, key: str) -> RepoGraph:
    with open(os.path.join(data_dir, "graphs", key + ".json"), encoding="utf-8") as f:
        g = json.load(f)
    ids = [n["id"] for n in g["nodes"]]
    text = [n.get("text") or "" for n in g["nodes"]]
    idx = {k: i for i, k in enumerate(ids)}
    adj: list[set[int]] = [set() for _ in ids]
    succ: list[set[int]] = [set() for _ in ids]
    for e in g["edges"]:
        s, t = idx.get(e["source"]), idx.get(e["target"])
        if s is None or t is None or s == t:
            continue
        adj[s].add(t)
        adj[t].add(s)
        succ[s].add(t)
    nbrs = [np.fromiter(sorted(a), dtype=np.int64, count=len(a)) for a in adj]

    emb_path = os.path.join(data_dir, "embeddings", key + ".npz")
    d = 256
    emb = np.zeros((len(ids), d), dtype=np.float32)
    has = np.zeros(len(ids), dtype=bool)
    if os.path.exists(emb_path):
        z = np.load(emb_path, allow_pickle=False)
        for k in z.files:
            i = idx.get(k)
            if i is not None:
                v = z[k].astype(np.float32).reshape(-1)
                nv = float(np.linalg.norm(v))
                if nv > 0 and v.shape[0] == d:
                    emb[i] = v / nv
                    has[i] = True

    cost = np.zeros((len(ids), 3), dtype=np.int64)
    for i, (nid, t) in enumerate(zip(ids, text)):
        head = est_tokens(nid) + 3  # "### <id>" header line
        cost[i, 0] = head
        cost[i, 1] = head + est_tokens(signature_text(t))
        cost[i, 2] = head + est_tokens(t)
    cost[:, 1] = np.maximum(cost[:, 1], cost[:, 0] + 1)
    cost[:, 2] = np.maximum(cost[:, 2], cost[:, 1])
    pool = np.array([not is_test_node(n) for n in ids], dtype=bool)
    return RepoGraph(key, ids, text, idx, nbrs, emb, has, cost, pool, succ=succ)

In [ ]:
%%writefile evtok/gold.py
"""Map a reference (gold) patch onto graph nodes. Used ONLY for evaluation labels and the oracle.

Graph nodes carry no file paths or line numbers, so mapping is textual:
  1. removed ('-') lines of a hunk -> nodes whose source contains that line (most specific id);
  2. hunks with no removed lines -> context lines adjacent to the added block;
  3. fallback -> function named in the hunk header (git's nearest preceding def).
Candidates are restricted to nodes whose dotted id matches the patched file's module path when possible.
"""
from __future__ import annotations

import re
from collections import defaultdict

from .data import RepoGraph
from .textproc import norm_line

_HDR = re.compile(r"^@@[^@]*@@\s*(?:async\s+)?(?:def|class)\s+(\w+)")
TRIVIAL = {"return", "pass", "else:", "try:", "finally:", ")", "]", "}", "),", "return None", "continue", "break"}


def parse_patch(patch: str) -> list[dict]:
    """Return [{'path': str, 'new_file': bool, 'hunks': [{'header': str, 'lines': [(op, text)]}]}]."""
    files: list[dict] = []
    cur = None
    hunk = None
    for line in patch.split("\n"):
        if line.startswith("--- "):
            cur = {"path": None, "new_file": line.startswith("--- /dev/null"), "hunks": []}
            files.append(cur)
            hunk = None
        elif line.startswith("+++ ") and cur is not None:
            p = line[4:].strip()
            cur["path"] = p[2:] if p.startswith("b/") else p
        elif line.startswith("@@") and cur is not None:
            hunk = {"header": line, "lines": []}
            cur["hunks"].append(hunk)
            if line.startswith("@@ -0,0 "):
                cur["new_file"] = True
        elif hunk is not None and line[:1] in ("+", "-", " "):
            hunk["lines"].append((line[:1], line[1:]))
    return [f for f in files if f["path"]]


def module_prefixes(path: str) -> list[str]:
    if not path.endswith(".py"):
        return []
    dotted = path[:-3].replace("/", ".")
    out = [dotted]
    parts = dotted.split(".")
    if parts[0] in ("src", "docs_src", "lib"):
        out.append(".".join(parts[1:]))
    if dotted.endswith(".__init__"):
        out = [p[: -len(".__init__")] for p in out]
    return [p for p in out if p]


def nodes_in_file(g: RepoGraph, path: str) -> set[int]:
    prefs = module_prefixes(path)
    is_init = path.endswith("__init__.py")
    out = set()
    for i, nid in enumerate(g.ids):
        for p in prefs:
            if nid == p or nid.startswith(p + "."):
                if is_init:
                    rest = nid[len(p) + 1:]
                    # __init__ members are p.name or p.Class.method; exclude submodules (lowercase.x.y)
                    if rest.count(".") >= 2:
                        continue
                out.add(i)
                break
    return out


def _line_index(g: RepoGraph) -> dict[str, set[int]]:
    if "line_index" not in g.extra:
        idx: dict[str, set[int]] = defaultdict(set)
        for i, t in enumerate(g.text):
            for l in t.split("\n"):
                n = norm_line(l)
                if len(n) >= 8 and n not in TRIVIAL:
                    idx[n].add(i)
        g.extra["line_index"] = idx
    return g.extra["line_index"]


def _most_specific(g: RepoGraph, cands: set[int]) -> set[int]:
    if not cands:
        return set()
    # prefer the deepest ids (method over class over module-level)
    best = max(g.ids[c].count(".") for c in cands)
    deep = {c for c in cands if g.ids[c].count(".") == best}
    # if one candidate's id is a prefix of another, keep the longer
    return {c for c in deep if not any(o != c and g.ids[o].startswith(g.ids[c] + ".") for o in deep)}


def gold_nodes(g: RepoGraph, patch: str) -> dict:
    """Return {'nodes': set[int], 'files': list[str], 'file_nodes': set[int], 'addition_only': bool}."""
    li = _line_index(g)
    gold: set[int] = set()
    file_nodes: set[int] = set()
    files = parse_patch(patch)
    for f in files:
        in_file = nodes_in_file(g, f["path"])
        file_nodes |= in_file
        if f["new_file"]:
            continue
        for h in f["hunks"]:
            hits: set[int] = set()
            lines = h["lines"]
            removed = [t for op, t in lines if op == "-"]
            probe = list(removed)
            if not removed:
                for k, (op, t) in enumerate(lines):
                    if op == "+":
                        if k > 0 and lines[k - 1][0] == " ":
                            probe.append(lines[k - 1][1])
                        if k + 1 < len(lines) and lines[k + 1][0] == " ":
                            probe.append(lines[k + 1][1])
            for t in probe:
                n = norm_line(t)
                if len(n) < 8 or n in TRIVIAL:
                    continue
                c = li.get(n, set())
                if in_file:
                    c = c & in_file
                hits |= _most_specific(g, c)
            if not hits:
                m = _HDR.match(h["header"])
                if m:
                    name = m.group(1)
                    pool = in_file if in_file else range(len(g.ids))
                    c = {i for i in pool if g.ids[i].rsplit(".", 1)[-1] == name}
                    if len(c) <= 3:
                        hits |= c
            gold |= hits
    return {
        "nodes": gold,
        "files": [f["path"] for f in files],
        "file_nodes": file_nodes,
        "addition_only": not gold,
    }

In [ ]:
%%writefile evtok/retrieval.py
"""Leakage-free relevance signals from the problem statement only.

- BM25 over identifier-split node id + source text.
- Identifier anchoring: names mentioned in the PR text resolved to graph nodes
  (same resolution order as the harness: exact -> dotted suffix -> last component).
- Dense pseudo-query: mean embedding of anchor nodes (or of BM25 top-m when no anchor resolves),
  scored by cosine similarity. No text encoder exists for the 256-d node space.
"""
from __future__ import annotations

import math
import re
from collections import Counter

import numpy as np

from .data import RepoGraph
from .textproc import STOP, clean_problem, ident_tokens


class BM25:
    def __init__(self, docs: list[list[str]], k1: float = 1.2, b: float = 0.75):
        self.k1, self.b = k1, b
        self.n = len(docs)
        self.dl = np.array([len(d) for d in docs], dtype=np.float32)
        self.avgdl = float(self.dl.mean()) if self.n else 1.0
        self.tf: list[Counter] = [Counter(d) for d in docs]
        df: Counter = Counter()
        for c in self.tf:
            df.update(c.keys())
        self.idf = {t: math.log(1 + (self.n - v + 0.5) / (v + 0.5)) for t, v in df.items()}
        self.post: dict[str, list[tuple[int, int]]] = {}
        for i, c in enumerate(self.tf):
            for t, v in c.items():
                self.post.setdefault(t, []).append((i, v))

    def score(self, q: list[str]) -> np.ndarray:
        s = np.zeros(self.n, dtype=np.float32)
        norm = self.k1 * (1 - self.b + self.b * self.dl / self.avgdl)
        for t, qtf in Counter(q).items():
            if t not in self.idf:
                continue
            idf = self.idf[t]
            for i, v in self.post[t]:
                s[i] += idf * v * (self.k1 + 1) / (v + norm[i])
        return s


def bm25_for_graph(g: RepoGraph) -> BM25:
    if "bm25" not in g.extra:
        docs = []
        for nid, t in zip(g.ids, g.text):
            idt = ident_tokens(nid.replace(".", " "))
            docs.append(idt + idt + ident_tokens(t[:20000]))
        g.extra["bm25"] = BM25(docs)
    return g.extra["bm25"]


_BT = re.compile(r"`([^`\n]{2,80})`")
_DOTTED = re.compile(r"\b[A-Za-z_]\w*(?:\.[A-Za-z_]\w*)+\b")
_CAMEL = re.compile(r"\b[A-Z][a-z0-9]+(?:[A-Z][a-z0-9]*)+\b")
_SNAKE = re.compile(r"\b[a-z][a-z0-9]*_[a-z0-9_]+\b")
_CALL = re.compile(r"\b([A-Za-z_]\w{2,})\(")


def extract_identifiers(ps: str) -> list[str]:
    ps = clean_problem(ps)
    cands: list[str] = []
    for m in _BT.findall(ps):
        cands.extend(re.findall(r"[A-Za-z_][\w.]*", m))
    cands += _DOTTED.findall(ps) + _CAMEL.findall(ps) + _SNAKE.findall(ps) + _CALL.findall(ps)
    out, seen = [], set()
    for c in cands:
        c = c.strip(".")
        if len(c) < 3 or c.lower() in STOP or c in seen:
            continue
        if re.fullmatch(r"[a-z]+\.(py|md|txt|toml|cfg|yaml|yml|json|html)", c):
            continue
        seen.add(c)
        out.append(c)
    return out


def resolve_anchors(g: RepoGraph, names: list[str], max_matches: int = 8) -> dict[int, float]:
    if "by_last" not in g.extra:
        by_last: dict[str, list[int]] = {}
        for i, nid in enumerate(g.ids):
            by_last.setdefault(nid.rsplit(".", 1)[-1], []).append(i)
        g.extra["by_last"] = by_last
    by_last = g.extra["by_last"]
    w: dict[int, float] = {}
    for c in names:
        if c.endswith(".py"):
            c = c[:-3].replace("/", ".")
        m: list[int] = []
        if c in g.idx:
            m = [g.idx[c]]
        else:
            last = c.rsplit(".", 1)[-1]
            cand = by_last.get(last, [])
            if "." in c:
                m = [i for i in cand if g.ids[i].endswith("." + c)]
            if not m:
                m = list(cand)
            if not m and "." in c:
                # module path mention, e.g. fastapi.routing -> all nodes under it are too many; skip
                m = []
        m = [i for i in m if g.pool[i]]
        if 0 < len(m) <= max_matches:
            for i in m:
                w[i] = max(w.get(i, 0.0), 1.0 / len(m))
    return w


def zscore(x: np.ndarray, mask: np.ndarray) -> np.ndarray:
    v = x[mask]
    sd = float(v.std()) if v.size else 0.0
    z = (x - float(v.mean() if v.size else 0.0)) / (sd if sd > 1e-9 else 1.0)
    z[~mask] = -1e9
    return z


def task_signals(g: RepoGraph, problem: str, m_centroid: int = 5) -> dict:
    """All per-task relevance signals, computed from the problem statement only."""
    q = ident_tokens(clean_problem(problem))
    bm = bm25_for_graph(g).score(q)
    bm[~g.pool] = 0.0
    names = extract_identifiers(problem)
    anchors = resolve_anchors(g, names)
    anc = np.zeros(len(g.ids), dtype=np.float32)
    for i, w in anchors.items():
        anc[i] = w

    def dense_from(nodes: list[int]) -> np.ndarray:
        nodes = [i for i in nodes if g.has_emb[i]]
        if not nodes:
            return np.zeros(len(g.ids), dtype=np.float32)
        qv = g.emb[nodes].mean(axis=0)
        n = float(np.linalg.norm(qv))
        return (g.emb @ (qv / n)) if n > 0 else np.zeros(len(g.ids), dtype=np.float32)

    order = np.argsort(-bm, kind="stable")
    top_bm = [int(i) for i in order[:m_centroid] if bm[i] > 0]
    dense_bm = dense_from(top_bm)
    dense_anchor = dense_from(list(anchors.keys())) if anchors else dense_bm
    return {
        "bm25": bm,
        "anchor": anc,
        "dense_anchor": dense_anchor,
        "dense_bm": dense_bm,
        "n_anchor": len(anchors),
        "names": names,
    }


def hybrid_score(g: RepoGraph, sig: dict, w_dense: float, w_anchor: float, dense_src: str = "anchor") -> np.ndarray:
    mask = g.pool
    s = zscore(np.log1p(sig["bm25"]), mask)
    if w_dense:
        d = sig["dense_anchor"] if dense_src == "anchor" else sig["dense_bm"]
        s = s + w_dense * zscore(d, mask)
    if w_anchor:
        s = s + w_anchor * sig["anchor"] * 3.0
    s[~mask] = -1e9
    return s

In [ ]:
%%writefile evtok/selectors.py
"""Context selectors under a hard token budget B.

A packet is a dict {node_index: tier} with tier in {0: outline, 1: signature+docstring, 2: full source}.
Its cost is sum(cost[v, tier]); every selector guarantees cost <= B.

Selectors
- flat_full:        nodes in score order, full source only (top-k retrieval at matched budget).
- greedy_tiers:     greedy on marginal utility per token over (node, tier-upgrade) actions.
                    expand=False -> flat multi-resolution packing (no graph).
                    expand=True  -> seed-rooted connected selection: a node becomes eligible only
                                    after one of its graph neighbours is selected (or it is a seed).
- graph_expand:     h-hop neighbourhood of seeds re-ranked with a proximity bonus, full source only.
- random_connected: random growth from the same seeds (control).
- oracle:           gold nodes first (full), then filled from another packet (diagnostic upper bound).
"""
from __future__ import annotations

import heapq
from collections import deque

import numpy as np

from .data import RepoGraph

TIER_W = (0.1, 0.45, 1.0)


def packet_cost(g: RepoGraph, pk: dict[int, int]) -> int:
    return int(sum(g.cost[v, t] for v, t in pk.items()))


def flat_full(g: RepoGraph, order, B: int, max_scan: int = 3000) -> dict[int, int]:
    pk: dict[int, int] = {}
    used = 0
    for n, v in enumerate(order):
        if n >= max_scan or B - used < 8:
            break
        v = int(v)
        c = int(g.cost[v, 2])
        if used + c <= B:
            pk[v] = 2
            used += c
    return pk


def bfs_dist(g: RepoGraph, seeds: list[int], h: int) -> dict[int, int]:
    dist = {s: 0 for s in seeds}
    dq = deque(seeds)
    while dq:
        v = dq.popleft()
        if dist[v] >= h:
            continue
        for n in g.nbrs[v]:
            n = int(n)
            if n not in dist and g.pool[n]:
                dist[n] = dist[v] + 1
                dq.append(n)
    return dist


def utility(s: np.ndarray, dist: dict[int, int] | None, beta: float, tau: float, temp: float) -> np.ndarray:
    sp = s.astype(np.float64).copy()
    if dist and beta:
        for v, d in dist.items():
            sp[v] += beta * np.exp(-d / tau)
    smax = float(sp[sp > -1e8].max()) if np.any(sp > -1e8) else 0.0
    u = np.exp(np.clip((sp - smax) / temp, -60, 0))
    u[s <= -1e8] = 0.0
    return u


def greedy_tiers(
    g: RepoGraph,
    util: np.ndarray,
    B: int,
    candidates,
    tiers=(0, 1, 2),
    rho: float = 1.0,
    expand: bool = False,
    region: dict[int, int] | None = None,
    wt=TIER_W,
) -> dict[int, int]:
    cur: dict[int, int] = {}
    heap: list = []
    cnt = 0
    pushed: set[int] = set()

    def push(v: int):
        nonlocal cnt
        c0 = cur.get(v, -1)
        w0 = wt[c0] if c0 >= 0 else 0.0
        k0 = int(g.cost[v, c0]) if c0 >= 0 else 0
        for t in tiers:
            if t <= c0:
                continue
            gain = util[v] * (wt[t] - w0)
            dc = int(g.cost[v, t]) - k0
            if gain <= 0 or dc <= 0:
                continue
            heapq.heappush(heap, (-gain / dc**rho, cnt, v, c0, t, dc))
            cnt += 1

    for v in candidates:
        v = int(v)
        if g.pool[v] and v not in pushed:
            pushed.add(v)
            push(v)
    used = 0
    while heap and B - used >= 4:
        _, _, v, c0, t, dc = heapq.heappop(heap)
        if cur.get(v, -1) != c0 or used + dc > B:
            continue
        cur[v] = t
        used += dc
        if c0 == -1 and expand:
            for n in g.nbrs[v]:
                n = int(n)
                if n in pushed or not g.pool[n] or (region is not None and n not in region):
                    continue
                pushed.add(n)
                push(n)
        push(v)
    return cur


def graph_expand(g: RepoGraph, s: np.ndarray, seeds: list[int], h: int, beta: float, tau: float, B: int) -> dict[int, int]:
    dist = bfs_dist(g, seeds, h)
    region = np.fromiter(dist.keys(), dtype=np.int64)
    sp = s[region] + beta * np.exp(-np.fromiter(dist.values(), dtype=np.float64) / tau)
    order = region[np.argsort(-sp, kind="stable")]
    return flat_full(g, order, B)


def random_connected(g: RepoGraph, seeds: list[int], B: int, rng: np.random.Generator) -> dict[int, int]:
    pk: dict[int, int] = {}
    used = 0
    frontier = list(dict.fromkeys(seeds))
    seen = set(frontier)
    fails = 0
    while frontier and B - used >= 8 and fails < 200:
        v = frontier.pop(int(rng.integers(len(frontier))))
        c = int(g.cost[v, 2])
        if used + c > B:
            fails += 1
            continue
        pk[v] = 2
        used += c
        for n in g.nbrs[v]:
            n = int(n)
            if n not in seen and g.pool[n]:
                seen.add(n)
                frontier.append(n)
    return pk


def oracle(g: RepoGraph, gold: set[int], fill: dict[int, int], B: int) -> dict[int, int]:
    pk: dict[int, int] = {}
    used = 0
    for v in sorted(gold, key=lambda x: int(g.cost[x, 2])):
        c = int(g.cost[v, 2])
        if used + c <= B:
            pk[v] = 2
            used += c
    for v, t in fill.items():
        if v in pk:
            continue
        for tt in range(t, -1, -1):
            c = int(g.cost[v, tt])
            if used + c <= B:
                pk[v] = tt
                used += c
                break
    return pk

In [ ]:
%%writefile evtok/metrics.py
"""Localization metrics for a packet against gold nodes (evaluation only)."""
from __future__ import annotations

import numpy as np

from .data import RepoGraph


def packet_metrics(g: RepoGraph, pk: dict[int, int], gold: dict) -> dict:
    G = gold["nodes"]
    full = {v for v, t in pk.items() if t == 2}
    anyt = set(pk)
    cost = sum(int(g.cost[v, t]) for v, t in pk.items())
    m = {
        "cost": cost,
        "n_nodes": len(pk),
        "n_full": len(full),
        "file_hit": float(bool(full & gold["file_nodes"])) if gold["file_nodes"] else float("nan"),
        "file_hit_any": float(bool(anyt & gold["file_nodes"])) if gold["file_nodes"] else float("nan"),
    }
    if G:
        hit_full = G & full
        m["recall"] = len(hit_full) / len(G)
        m["recall_any"] = len(G & anyt) / len(G)
        m["hit"] = float(bool(hit_full))
        m["hit_any"] = float(bool(G & anyt))
        m["density"] = (sum(int(g.cost[v, 2]) for v in hit_full) / cost) if cost else 0.0
    else:
        for k in ("recall", "recall_any", "hit", "hit_any", "density"):
            m[k] = float("nan")
    # connectivity diagnostics
    sel = set(pk)
    comps = 0
    seen: set[int] = set()
    for v in sel:
        if v in seen:
            continue
        comps += 1
        stack = [v]
        seen.add(v)
        while stack:
            x = stack.pop()
            for n in g.nbrs[x]:
                n = int(n)
                if n in sel and n not in seen:
                    seen.add(n)
                    stack.append(n)
    m["components"] = comps
    return m


def recall_auc(values_by_budget: list[float]) -> float:
    """Mean over log-spaced budgets (equal weight per doubling)."""
    v = [x for x in values_by_budget if not np.isnan(x)]
    return float(np.mean(v)) if v else float("nan")

In [ ]:
%%writefile evtok/run_tier_a.py
"""Tier A: evaluate every selector configuration on every task at every budget (CPU only).

Usage:
    python -m evtok.run_tier_a --data <competition data dir> --out results/tier_a [--workers 4]

Writes <out>/grid.pkl: {instance_id: {"meta": {...}, "res": {config_key: {budget: metrics}}}}.
Hyper-parameter selection (5-fold CV) and statistics are done afterwards by evtok.analyze;
nothing here looks at gold labels except to compute metrics after a packet is built.
"""
from __future__ import annotations

import argparse
import itertools
import os
import pickle
import time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

from . import selectors as S
from .data import graph_key, load_graph, load_tasks
from .gold import gold_nodes
from .metrics import packet_metrics
from .retrieval import hybrid_score, task_signals, zscore

BUDGETS = (2000, 4000, 8000, 16000)
KEEP = ("recall", "recall_any", "hit", "hit_any", "density", "file_hit", "file_hit_any", "cost", "n_nodes", "n_full", "components")

HYBRIDS = [(0.0, 0.0, "anchor")] + [
    (wd, wa, src) for wd in (0.0, 0.5, 1.0) for wa in (0.0, 1.0, 2.0) for src in ("anchor", "bm") if not (wd == 0.0 and src == "bm") and (wd, wa) != (0.0, 0.0)
]
TIERW = {"a": (0.02, 0.2, 1.0), "b": (0.1, 0.45, 1.0)}
CONN_GRID = list(itertools.product((0.0, 1.0), (1, 2), (3, 5, 10), (0.5, 1.0), ("a", "b")))  # beta,h,K,rho,tierw
GX_GRID = list(itertools.product((0.0, 1.0), (1, 2), (3, 5, 10)))  # beta,h,K
FLAT_TIER_GRID = list(itertools.product((0.5, 1.0), ("a", "b")))
TAU, TEMP = 1.0, 1.5


def hkey(h):
    return f"wd{h[0]}_wa{h[1]}_{h[2]}"


def _m(g, pk, gold):
    m = packet_metrics(g, pk, gold)
    return {k: m[k] for k in KEEP}


def run_group(data_dir: str, key: str, tasks: list[dict], seed: int = 0) -> dict:
    g = load_graph(data_dir, key)
    out = {}
    for t in tasks:
        gold = gold_nodes(g, t["patch"])
        sig = task_signals(g, t["problem_statement"])
        res: dict[str, dict[int, dict]] = defaultdict(dict)
        # pure dense baselines
        for name, d in (("dense_anchor", sig["dense_anchor"]), ("dense_bm", sig["dense_bm"])):
            s = zscore(d.astype(np.float64), g.pool)
            order = np.argsort(-s, kind="stable")
            for B in BUDGETS:
                res[f"flat|{name}"][B] = _m(g, S.flat_full(g, order, B), gold)
        rng = np.random.default_rng(seed)
        for h in HYBRIDS:
            hk = hkey(h)
            s = hybrid_score(g, sig, *h)
            order = np.argsort(-s, kind="stable")
            top = [int(v) for v in order[:400] if s[v] > -1e8]
            seeds_by_k = {k: top[:k] for k in (3, 5, 10)}
            u_flat = S.utility(s, None, 0.0, TAU, TEMP)
            dists = {(k, hh): S.bfs_dist(g, seeds_by_k[k], hh) for k in (3, 5, 10) for hh in (1, 2)}
            utils = {(k, hh, beta): S.utility(s, dists[(k, hh)], beta, TAU, TEMP) for k in (3, 5, 10) for hh in (1, 2) for beta in (0.0, 1.0)}
            for B in BUDGETS:
                ff = S.flat_full(g, order, B)
                res[f"flat|{hk}"][B] = _m(g, ff, gold)
                for rho, tw in FLAT_TIER_GRID:
                    pk = S.greedy_tiers(g, u_flat, B, top, rho=rho, expand=False, wt=TIERW[tw])
                    res[f"flatT|{hk}|rho{rho}_{tw}"][B] = _m(g, pk, gold)
                for beta, hh, k in GX_GRID:
                    pk = S.graph_expand(g, s, seeds_by_k[k], hh, beta, TAU, B)
                    res[f"gx|{hk}|b{beta}_h{hh}_k{k}"][B] = _m(g, pk, gold)
                    pk = S.greedy_tiers(g, utils[(k, hh, beta)], B, seeds_by_k[k], tiers=(2,), rho=1.0, expand=True, region=dists[(k, hh)])
                    res[f"conn|{hk}|b{beta}_h{hh}_k{k}"][B] = _m(g, pk, gold)
                for beta, hh, k, rho, tw in CONN_GRID:
                    pk = S.greedy_tiers(g, utils[(k, hh, beta)], B, seeds_by_k[k], rho=rho, expand=True, region=dists[(k, hh)], wt=TIERW[tw])
                    res[f"connT|{hk}|b{beta}_h{hh}_k{k}_rho{rho}_{tw}"][B] = _m(g, pk, gold)
                # random connected control (3 draws, averaged later), seeds = top-5
                for r in range(3):
                    pk = S.random_connected(g, seeds_by_k[5], B, rng)
                    res[f"rand|{hk}|r{r}"][B] = _m(g, pk, gold)
                res[f"oracle|{hk}"][B] = _m(g, S.oracle(g, gold["nodes"], ff, B), gold)
        meta = {
            "repo": t["repo"],
            "graph": key,
            "n_gold": len(gold["nodes"]),
            "gold_ids": sorted(g.ids[v] for v in gold["nodes"]),
            "gold_in_pool": all(g.pool[v] for v in gold["nodes"]),
            "addition_only": gold["addition_only"],
            "n_files": len(gold["files"]),
            "n_anchor": sig["n_anchor"],
            "n_nodes_graph": len(g.ids),
            "n_pool": int(g.pool.sum()),
            "gold_cost_full": int(sum(g.cost[v, 2] for v in gold["nodes"])),
        }
        out[t["instance_id"]] = {"meta": meta, "res": dict(res)}
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results/tier_a")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--limit", type=int, default=0, help="debug: only first N graph groups")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tasks = load_tasks(a.data)
    groups: dict[str, list[dict]] = defaultdict(list)
    for t in tasks:
        groups[graph_key(t)].append(t)
    keys = sorted(groups)
    if a.limit:
        keys = keys[: a.limit]
    t0 = time.time()
    allres: dict = {}
    with ProcessPoolExecutor(max_workers=a.workers) as ex:
        futs = {ex.submit(run_group, a.data, k, groups[k]): k for k in keys}
        for n, f in enumerate(as_completed(futs), 1):
            allres.update(f.result())
            print(f"[{n}/{len(keys)}] {futs[f]}  {time.time() - t0:.0f}s", flush=True)
    with open(os.path.join(a.out, "grid.pkl"), "wb") as f:
        pickle.dump({"budgets": BUDGETS, "results": allres}, f)
    print("done", len(allres), "tasks", f"{time.time() - t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evtok/analyze.py
"""Tier A analysis: 5-fold task-level CV config selection, out-of-fold metrics, paired statistics.

Usage:
    python -m evtok.analyze --grid results/tier_a/grid.pkl --out results/tier_a

Selection objective (training folds only): mean over tasks with gold nodes of
mean-over-budgets function Recall@B (full-source tier).
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import os
import pickle
from collections import defaultdict

import numpy as np

METHODS = [
    # name, config-key prefix, tuned?
    ("BM25 top-k", "flat|wd0.0_wa0.0_anchor", False),
    ("Dense top-k (anchor query)", "flat|dense_anchor", False),
    ("Dense top-k (BM25-centroid query)", "flat|dense_bm", False),
    ("Hybrid top-k", "flat|", True),
    ("Hybrid + tiers (no graph)", "flatT|", True),
    ("Graph expansion (full only)", "gx|", True),
    ("Connected (full only)", "conn|", True),
    ("Connected + tiers (EpT)", "connT|", True),
]
DEFAULTS = {
    "Hybrid top-k [default]": "flat|wd0.5_wa1.0_anchor",
    "Connected + tiers (EpT) [default]": "connT|wd0.5_wa1.0_anchor|b1.0_h2_k5_rho1.0_b",
}
BASELINE = "Hybrid top-k"


def folds_for(tasks: dict, k: int = 5, seed: int = 0) -> dict[str, int]:
    rng = np.random.default_rng(seed)
    by_repo = defaultdict(list)
    for iid, r in sorted(tasks.items()):
        by_repo[r["meta"]["repo"]].append(iid)
    f = {}
    off = 0
    for repo in sorted(by_repo):
        ids = by_repo[repo]
        rng.shuffle(ids)
        for i, iid in enumerate(ids):
            f[iid] = (i + off) % k
        off += len(ids)
    return f


def auc(res_cfg: dict, budgets, key="recall") -> float:
    v = [res_cfg[B][key] for B in budgets]
    v = [x for x in v if not (isinstance(x, float) and math.isnan(x))]
    return float(np.mean(v)) if v else float("nan")


def candidate_keys(sample_res: dict, prefix: str, tuned: bool) -> list[str]:
    if not tuned:
        return [prefix] if prefix in sample_res else []
    ks = [k for k in sample_res if k.startswith(prefix)]
    if prefix == "flat|":
        ks = [k for k in ks if not k.startswith("flat|dense")]
    return sorted(ks)


def select(train_ids, R, keys, budgets) -> str:
    best, bv = keys[0], -1.0
    for k in keys:
        v = [auc(R[i]["res"][k], budgets) for i in train_ids]
        v = [x for x in v if not math.isnan(x)]
        m = float(np.mean(v)) if v else -1.0
        if m > bv + 1e-12:
            best, bv = k, m
    return best


def mcnemar_exact(b: int, c: int) -> float:
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2**n
    return min(1.0, 2 * p)


def boot_ci(diff: np.ndarray, n_boot: int = 10000, seed: int = 0):
    rng = np.random.default_rng(seed)
    diff = diff[~np.isnan(diff)]
    if diff.size == 0:
        return (float("nan"),) * 3
    idx = rng.integers(0, diff.size, size=(n_boot, diff.size))
    means = diff[idx].mean(axis=1)
    return float(diff.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--grid", default="results/tier_a/grid.pkl")
    ap.add_argument("--out", default="results/tier_a")
    a = ap.parse_args()
    import sys
    sys.stdout.reconfigure(encoding="utf-8")
    G = pickle.load(open(a.grid, "rb"))
    budgets = list(G["budgets"])
    R = G["results"]
    ids = sorted(R)
    fold = folds_for(R)
    sample = R[ids[0]]["res"]
    eval_ids = [i for i in ids if R[i]["meta"]["n_gold"] > 0]

    oof: dict[str, dict[str, dict]] = defaultdict(dict)  # method -> iid -> {B: metrics}
    choices: dict[str, list] = defaultdict(list)
    for name, prefix, tuned in METHODS:
        keys = candidate_keys(sample, prefix, tuned)
        for f in range(5):
            train = [i for i in eval_ids if fold[i] != f]
            test = [i for i in ids if fold[i] == f]
            k = select(train, R, keys, budgets) if tuned else keys[0]
            choices[name].append(k)
            for i in test:
                oof[name][i] = R[i]["res"][k]
            if name == "Connected + tiers (EpT)":
                hk = k.split("|")[1]
                for i in test:
                    draws = [R[i]["res"][f"rand|{hk}|r{r}"] for r in range(3)]
                    oof["Random connected (control)"][i] = {
                        B: {m: float(np.nanmean([d[B][m] for d in draws])) for m in draws[0][B]} for B in budgets
                    }
            if name == "Hybrid top-k":
                hk = k.split("|")[1]
                for i in test:
                    oof["Oracle (gold first)"][i] = R[i]["res"][f"oracle|{hk}"]
    for name, k in DEFAULTS.items():
        for i in ids:
            oof[name][i] = R[i]["res"][k]

    order = [m[0] for m in METHODS] + ["Random connected (control)", "Oracle (gold first)"] + list(DEFAULTS)
    lines = []
    lines.append(f"# Tier A results (out-of-fold, 5-fold task-level CV)\n")
    lines.append(f"Tasks: {len(ids)}; with gold function nodes: {len(eval_ids)}; addition-only: {len(ids) - len(eval_ids)}.\n")
    # main table
    hdr = "| Method | " + " | ".join(f"Recall@{B//1000}k" for B in budgets) + " | Recall-AUC | Hit@4k | Hit@8k | AnyTier@4k | File@4k |"
    lines.append(hdr)
    lines.append("|" + "---|" * (hdr.count("|") - 1))
    rows_csv = []
    for name in order:
        per = oof[name]
        rec = [np.nanmean([per[i][B]["recall"] for i in eval_ids]) for B in budgets]
        aucv = np.nanmean([auc(per[i], budgets) for i in eval_ids])
        h4 = np.nanmean([per[i][4000]["hit"] for i in eval_ids])
        h8 = np.nanmean([per[i][8000]["hit"] for i in eval_ids])
        a4 = np.nanmean([per[i][4000]["recall_any"] for i in eval_ids])
        f4 = np.nanmean([per[i][4000]["file_hit"] for i in ids if not math.isnan(per[i][4000]["file_hit"])])
        lines.append(f"| {name} | " + " | ".join(f"{x:.3f}" for x in rec) + f" | {aucv:.3f} | {h4:.3f} | {h8:.3f} | {a4:.3f} | {f4:.3f} |")
        rows_csv.append([name] + [f"{x:.4f}" for x in rec] + [f"{aucv:.4f}", f"{h4:.4f}", f"{h8:.4f}", f"{a4:.4f}", f"{f4:.4f}"])
    # paired stats vs baseline
    lines.append(f"\n## Paired differences vs {BASELINE} (bootstrap 95% CI over {len(eval_ids)} tasks; McNemar exact on Hit)\n")
    lines.append("| Method | ΔRecall-AUC [95% CI] | " + " | ".join(f"ΔRecall@{B//1000}k [CI]" for B in budgets) + " | Hit@4k b/c, p | Hit@8k b/c, p |")
    lines.append("|" + "---|" * (len(budgets) + 4))
    base = oof[BASELINE]
    stats = {}
    for name in order:
        if name == BASELINE:
            continue
        per = oof[name]
        d_auc = np.array([auc(per[i], budgets) - auc(base[i], budgets) for i in eval_ids])
        cells = []
        m, lo, hi = boot_ci(d_auc)
        cells.append(f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]")
        st = {"auc": (m, lo, hi)}
        for B in budgets:
            d = np.array([per[i][B]["recall"] - base[i][B]["recall"] for i in eval_ids])
            m, lo, hi = boot_ci(d)
            cells.append(f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]")
            st[B] = (m, lo, hi)
        for B in (4000, 8000):
            b = sum(per[i][B]["hit"] == 1 and base[i][B]["hit"] == 0 for i in eval_ids)
            c = sum(per[i][B]["hit"] == 0 and base[i][B]["hit"] == 1 for i in eval_ids)
            cells.append(f"{b}/{c}, p={mcnemar_exact(b, c):.3f}")
            st[f"mcnemar{B}"] = (b, c, mcnemar_exact(b, c))
        stats[name] = st
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    # per-repo AUC
    repos = sorted({R[i]["meta"]["repo"] for i in eval_ids})
    lines.append("\n## Recall-AUC by repository\n")
    lines.append("| Method | " + " | ".join(f"{r} (n={sum(R[i]['meta']['repo']==r for i in eval_ids)})" for r in repos) + " |")
    lines.append("|" + "---|" * (len(repos) + 1))
    for name in order:
        per = oof[name]
        lines.append(f"| {name} | " + " | ".join(f"{np.nanmean([auc(per[i], budgets) for i in eval_ids if R[i]['meta']['repo']==r]):.3f}" for r in repos) + " |")
    # packet shape
    lines.append("\n## Packet composition at 8k (mean)\n")
    lines.append("| Method | nodes | full-source nodes | components | cost | evidence density |")
    lines.append("|---|---|---|---|---|---|")
    for name in order:
        per = oof[name]
        g = lambda k: np.nanmean([per[i][8000][k] for i in eval_ids])
        lines.append(f"| {name} | {g('n_nodes'):.1f} | {g('n_full'):.1f} | {g('components'):.1f} | {g('cost'):.0f} | {g('density'):.3f} |")
    lines.append("\n## CV choices per fold\n")
    for name in order:
        if name in choices:
            lines.append(f"- {name}: " + "; ".join(choices[name]))
    os.makedirs(a.out, exist_ok=True)
    open(os.path.join(a.out, "summary.md"), "w", encoding="utf-8").write("\n".join(lines) + "\n")
    with open(os.path.join(a.out, "summary.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["method"] + [f"recall@{B}" for B in budgets] + ["recall_auc", "hit@4k", "hit@8k", "anytier@4k", "file@4k"])
        w.writerows(rows_csv)
    with open(os.path.join(a.out, "oof_per_task.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["instance_id", "repo", "fold", "n_gold", "method", "budget", "recall", "recall_any", "hit", "file_hit", "cost", "n_nodes", "components"])
        for name in order:
            for i in ids:
                for B in budgets:
                    m = oof[name][i][B]
                    w.writerow([i, R[i]["meta"]["repo"], fold[i], R[i]["meta"]["n_gold"], name, B, m["recall"], m["recall_any"], m["hit"], m["file_hit"], m["cost"], m["n_nodes"], m["components"]])
    json.dump({"choices": choices, "folds": fold}, open(os.path.join(a.out, "cv_choices.json"), "w"), indent=1)
    print("\n".join(lines))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evtok/diagnostics.py
"""Structural headroom: could graph neighbourhoods of retrieved seeds reach the gold functions?

Usage: python -m evtok.diagnostics --data <DATA_DIR> --out results/tier_a
For each task with gold nodes: degree of gold nodes, hop distance from BM25 top-k seeds to the
nearest gold node, and whether BM25 top-k at 8k already hits gold.
"""
from __future__ import annotations

import argparse
import csv
import os
from collections import Counter, defaultdict, deque

import numpy as np

from . import selectors as S
from .data import graph_key, load_graph, load_tasks
from .gold import gold_nodes
from .retrieval import hybrid_score, task_signals


def hop(g, seeds, targets, hmax=4):
    dist = {s: 0 for s in seeds}
    dq = deque(seeds)
    while dq:
        v = dq.popleft()
        if v in targets:
            return dist[v]
        if dist[v] >= hmax:
            continue
        for n in g.nbrs[v]:
            n = int(n)
            if n not in dist:
                dist[n] = dist[v] + 1
                dq.append(n)
    return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results/tier_a")
    a = ap.parse_args()
    tasks = load_tasks(a.data)
    groups = defaultdict(list)
    for t in tasks:
        groups[graph_key(t)].append(t)
    rows = []
    for k in sorted(groups):
        g = load_graph(a.data, k)
        deg = np.array([len(n) for n in g.nbrs])
        for t in groups[k]:
            gd = gold_nodes(g, t["patch"])
            G = gd["nodes"]
            if not G:
                continue
            s = hybrid_score(g, task_signals(g, t["problem_statement"]), 0.0, 0.0)
            order = np.argsort(-s, kind="stable")
            ff = S.flat_full(g, order, 8000)
            hit8 = bool(set(ff) & G)
            rank = min(int(np.where(order == v)[0][0]) for v in G)
            rows.append({
                "instance_id": t["instance_id"], "repo": t["repo"], "n_gold": len(G),
                "gold_min_degree": int(min(deg[v] for v in G)), "gold_isolated_frac": float(np.mean([deg[v] == 0 for v in G])),
                "graph_isolated_frac": float(np.mean(deg[g.pool] == 0)),
                "best_gold_rank_bm25": rank, "bm25_hit8k": hit8,
                "hop_top1": hop(g, [int(order[0])], G), "hop_top10": hop(g, [int(v) for v in order[:10]], G),
                "hop_selected8k": hop(g, list(ff), G),
            })
    with open(os.path.join(a.out, "diagnostics.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    n = len(rows)
    miss = [r for r in rows if not r["bm25_hit8k"]]
    fmt = lambda c: dict(sorted(Counter("none(>4)" if x is None else x for x in c).items(), key=lambda kv: str(kv[0])))
    print(f"tasks with gold: {n}; BM25 misses at 8k: {len(miss)}")
    print("share of pool nodes with no edges (mean over tasks):", round(np.mean([r['graph_isolated_frac'] for r in rows]), 3))
    print("tasks where every gold node is isolated:", sum(r["gold_isolated_frac"] == 1 for r in rows))
    print("hop from BM25 top-1 to nearest gold (all):", fmt(r["hop_top1"] for r in rows))
    print("hop from BM25 top-10 to nearest gold (all):", fmt(r["hop_top10"] for r in rows))
    print("BM25 8k misses — hop from the selected 8k packet to nearest gold:", fmt(r["hop_selected8k"] for r in miss))
    print("BM25 8k misses — every gold node isolated:", sum(r["gold_isolated_frac"] == 1 for r in miss))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evtok/figures.py
"""Paper figures from Tier A out-of-fold results (static PNG for the Kaggle Writeup).

Usage: python -m evtok.figures --csv results/tier_a/oof_per_task.csv --out results/tier_a
"""
from __future__ import annotations

import argparse
import csv
import math
import os
from collections import defaultdict

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

# Reference categorical palette (light), fixed order; controls in neutral gray.
SERIES = [
    ("BM25 top-k", "#2a78d6", "-", "o"),
    ("Dense top-k (anchor query)", "#eb6834", "-", "s"),
    ("Hybrid top-k", "#1baf7a", "-", "D"),
    ("Hybrid + tiers (no graph)", "#eda100", "-", "^"),
    ("Connected + tiers (EpT)", "#6250d6", "-", "v"),
    ("Random connected (control)", "#8a8984", ":", "x"),
    ("Oracle (gold first)", "#52514e", "--", "+"),
]
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"


def load(path):
    d = defaultdict(lambda: defaultdict(list))
    with open(path, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if int(r["n_gold"]) == 0:
                continue
            v = float(r["recall"])
            if not math.isnan(v):
                d[r["method"]][int(r["budget"])].append(v)
    return d


def boot(v, n=4000, seed=0):
    v = np.asarray(v)
    rng = np.random.default_rng(seed)
    m = v[rng.integers(0, v.size, (n, v.size))].mean(1)
    return np.percentile(m, 2.5), np.percentile(m, 97.5)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="results/tier_a/oof_per_task.csv")
    ap.add_argument("--out", default="results/tier_a")
    a = ap.parse_args()
    d = load(a.csv)
    plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2})
    fig, ax = plt.subplots(figsize=(7.0, 4.2), dpi=200, facecolor=SURF)
    ax.set_facecolor(SURF)
    for name, col, ls, mk in SERIES:
        if name not in d:
            continue
        B = sorted(d[name])
        y = [np.mean(d[name][b]) for b in B]
        lo, hi = zip(*[boot(d[name][b]) for b in B])
        ax.fill_between(B, lo, hi, color=col, alpha=0.10, linewidth=0)
        ax.plot(B, y, ls, color=col, lw=2, marker=mk, ms=6, label=name, markeredgecolor=SURF, markeredgewidth=1)
    ax.set_xscale("log", base=2)
    ax.set_xticks([2000, 4000, 8000, 16000], ["2k", "4k", "8k", "16k"])
    ax.minorticks_off()
    ax.set_xlabel("Context budget B (estimated tokens)")
    ax.set_ylabel("Function Recall@B (gold functions in full source)")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", color=GRID, lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper left", labelcolor=INK)
    ax.set_title("Recall vs. context budget (out-of-fold, 120 tasks, 95% bootstrap bands)", fontsize=10, color=INK, loc="left")
    fig.tight_layout()
    p = os.path.join(a.out, "fig_recall_vs_budget.png")
    fig.savefig(p, facecolor=SURF)
    print("wrote", p)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evtok/extensions.py
"""Tier A extensions: learned candidate ranker, budget-split graph rescue, Steiner-connected top-k.

Usage:
    python -m evtok.extensions --data <DATA_DIR> --out results/tier_a_ext --workers 8

Pass 1 (parallel over graphs): per task, build candidates and features (problem statement, graph and
embeddings only), compute packets for the non-learned extension baselines, and store gold labels for
evaluation/training. Pass 2: 5-fold task-level CV (same folds as evtok.analyze); the ranker is trained only
on training-fold tasks and scored on held-out tasks; every reported number is out-of-fold.
"""
from __future__ import annotations

import argparse
import itertools
import json
import math
import os
import pickle
import time
from collections import defaultdict, deque
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

from . import selectors as S
from .analyze import boot_ci, folds_for, mcnemar_exact
from .data import graph_key, load_graph, load_tasks
from .gold import gold_nodes
from .metrics import packet_metrics
from .retrieval import hybrid_score, task_signals
from .run_tier_a import BUDGETS, KEEP, TAU, TEMP, TIERW
from .textproc import clean_problem, ident_tokens

FEATS = [
    "bm25_z", "bm25_logrank", "anchor", "dense_anchor_z", "dense_bm_z",
    "hop_top1", "hop_top3", "hop_top10", "n_adj_top10", "log_degree",
    "is_class", "depth", "log_len", "same_parent_top1", "share_prefix_top10",
    "name_in_query", "id_token_overlap", "in_bm25_top200",
]
RESCUE_GRID = list(itertools.product((0.25, 0.5), (3, 10), (1, 2)))  # frac, m seeds, rescue tier
STEINER_GRID = list(itertools.product((3, 5, 10), (0, 1)))  # K, connector tier


def multi_bfs(g, seeds, hmax=5):
    dist = {s: 0 for s in seeds}
    dq = deque(seeds)
    while dq:
        v = dq.popleft()
        if dist[v] >= hmax:
            continue
        for n in g.nbrs[v]:
            n = int(n)
            if n not in dist and g.pool[n]:
                dist[n] = dist[v] + 1
                dq.append(n)
    return dist


def zs(x, idx):
    v = x[idx]
    sd = v.std()
    return (x - v.mean()) / (sd if sd > 1e-9 else 1.0)


def task_features(g, sig):
    bm = np.log1p(sig["bm25"].astype(np.float64))
    pool_idx = np.where(g.pool)[0]
    order = pool_idx[np.argsort(-bm[pool_idx], kind="stable")]
    rank = {int(v): r for r, v in enumerate(order)}
    top = [int(v) for v in order[:10]]
    d1, d3, d10 = multi_bfs(g, top[:1]), multi_bfs(g, top[:3]), multi_bfs(g, top[:10])
    cand = list(dict.fromkeys([int(v) for v in order[:200]] + [v for v, d in multi_bfs(g, top, 2).items()]))
    bmz = zs(bm, pool_idx)
    daz = zs(sig["dense_anchor"].astype(np.float64), pool_idx)
    dbz = zs(sig["dense_bm"].astype(np.float64), pool_idx)
    qtok = set(ident_tokens(clean_problem(sig["_problem"])))
    parent1 = g.ids[top[0]].rsplit(".", 1)[0] if top else ""
    pref = lambda s: ".".join(s.split(".")[:2])
    top_pref = [pref(g.ids[v]) for v in top]
    topset = set(top)
    X = np.zeros((len(cand), len(FEATS)), dtype=np.float32)
    for r, v in enumerate(cand):
        nid = g.ids[v]
        last = nid.rsplit(".", 1)[-1]
        idt = set(ident_tokens(nid.replace(".", " ")))
        X[r] = [
            bmz[v], math.log1p(rank.get(v, 10**5)), sig["anchor"][v], daz[v], dbz[v],
            d1.get(v, 6), d3.get(v, 6), d10.get(v, 6), sum(int(n) in topset for n in g.nbrs[v]), math.log1p(len(g.nbrs[v])),
            float(g.text[v].lstrip().startswith("class ")), nid.count("."), math.log1p(len(g.text[v])),
            float(bool(parent1) and (nid.startswith(parent1 + ".") or nid.rsplit(".", 1)[0] == parent1)),
            sum(pref(nid) == p for p in top_pref) / max(1, len(top_pref)),
            float(last.lower() in qtok), (len(idt & qtok) / len(idt)) if idt else 0.0, float(rank.get(v, 10**5) < 200),
        ]
    return np.array(cand, dtype=np.int64), X, order, top


def rescue_packet(g, order, s, B, frac, m, tier):
    base = S.flat_full(g, order, int(B * (1 - frac)))
    used = S.packet_cost(g, base)
    seeds = [int(v) for v in order[:m]]
    nb = [int(n) for v in seeds for n in g.nbrs[v] if g.pool[int(n)] and int(n) not in base]
    nb = sorted(set(nb), key=lambda v: -s[v])
    pk = dict(base)
    for v in nb:
        for t in ((tier, 0) if tier == 1 else (2, 1, 0)):
            c = int(g.cost[v, t])
            if used + c <= B:
                pk[v] = t
                used += c
                break
    for v in order:  # fill any leftover with flat full source
        v = int(v)
        if B - used < 8:
            break
        if v not in pk and used + int(g.cost[v, 2]) <= B:
            pk[v] = 2
            used += int(g.cost[v, 2])
    return pk


def steiner_packet(g, order, B, K, conn_tier):
    """Connect the top-K seeds to the top-1 seed by shortest paths (<=3 hops); seeds full, connectors at conn_tier."""
    seeds = [int(v) for v in order[:K]]
    if not seeds:
        return {}
    root = seeds[0]
    parent = {root: None}
    dist = {root: 0}
    dq = deque([root])
    while dq:
        v = dq.popleft()
        if dist[v] >= 3:
            continue
        for n in g.nbrs[v]:
            n = int(n)
            if n not in dist and g.pool[n]:
                dist[n] = dist[v] + 1
                parent[n] = v
                dq.append(n)
    tree = set()
    for sd in seeds:
        x = sd if sd in parent else None
        while x is not None:
            tree.add(x)
            x = parent[x]
    tree.add(root)
    pk, used = {}, 0
    for v in seeds:  # seeds first, full source, in rank order
        c = int(g.cost[v, 2])
        if used + c <= B:
            pk[v] = 2
            used += c
    for v in tree - set(seeds):
        c = int(g.cost[v, conn_tier])
        if used + c <= B:
            pk[v] = conn_tier
            used += c
    for v in order:
        v = int(v)
        if B - used < 8:
            break
        if v not in pk and used + int(g.cost[v, 2]) <= B:
            pk[v] = 2
            used += int(g.cost[v, 2])
    return pk


def _m(g, pk, gold):
    m = packet_metrics(g, pk, gold)
    return {k: m[k] for k in KEEP}


def run_group(data_dir, key, tasks):
    g = load_graph(data_dir, key)
    out = {}
    for t in tasks:
        gold = gold_nodes(g, t["patch"])
        sig = task_signals(g, t["problem_statement"])
        sig["_problem"] = t["problem_statement"]
        cand, X, order, top = task_features(g, sig)
        s = hybrid_score(g, sig, 0.0, 0.0, "anchor")
        res = defaultdict(dict)
        for B in BUDGETS:
            res["bm25"][B] = _m(g, S.flat_full(g, order, B), gold)
            for frac, m, tier in RESCUE_GRID:
                res[f"rescue|f{frac}_m{m}_t{tier}"][B] = _m(g, rescue_packet(g, order, s, B, frac, m, tier), gold)
            for K, ct in STEINER_GRID:
                res[f"steiner|k{K}_t{ct}"][B] = _m(g, steiner_packet(g, order, B, K, ct), gold)
        y = np.array([int(v) in gold["nodes"] for v in cand], dtype=np.int8)
        out[t["instance_id"]] = {
            "repo": t["repo"], "graph": key, "n_gold": len(gold["nodes"]), "cand": cand, "X": X, "y": y,
            "gold_in_cand": bool(set(cand.tolist()) & gold["nodes"]) if gold["nodes"] else False,
            "res": dict(res),
        }
    return out


def score_learned(data_dir, groups, preds):
    """Build packets from learned scores for held-out tasks. preds: iid -> (cand, prob)."""
    out = {}
    for key, tasks in groups.items():
        tasks = [t for t in tasks if t["instance_id"] in preds]
        if not tasks:
            continue
        g = load_graph(data_dir, key)
        for t in tasks:
            gold = gold_nodes(g, t["patch"])
            cand, p = preds[t["instance_id"]]
            sc = np.full(len(g.ids), -1e9)
            sc[cand] = p
            order = cand[np.argsort(-p, kind="stable")]
            res = {"learned": {}, "learnedT": {}}
            u = np.zeros(len(g.ids))
            u[cand] = p / max(p.max(), 1e-9)
            for B in BUDGETS:
                res["learned"][B] = _m(g, S.flat_full(g, order, B), gold)
                res["learnedT"][B] = _m(g, S.greedy_tiers(g, u, B, cand, rho=0.5, expand=False, wt=TIERW["a"]), gold)
            out[t["instance_id"]] = res
    return out


def auc(r):
    v = [r[B]["recall"] for B in BUDGETS]
    v = [x for x in v if not (isinstance(x, float) and math.isnan(x))]
    return float(np.mean(v)) if v else float("nan")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results/tier_a_ext")
    ap.add_argument("--workers", type=int, default=4)
    a = ap.parse_args()
    import sys
    sys.stdout.reconfigure(encoding="utf-8")
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    os.makedirs(a.out, exist_ok=True)
    tasks = load_tasks(a.data)
    groups = defaultdict(list)
    for t in tasks:
        groups[graph_key(t)].append(t)
    t0 = time.time()
    R = {}
    with ProcessPoolExecutor(max_workers=a.workers) as ex:
        futs = [ex.submit(run_group, a.data, k, v) for k, v in sorted(groups.items())]
        for n, f in enumerate(as_completed(futs), 1):
            R.update(f.result())
            if n % 20 == 0:
                print(f"features {n}/{len(futs)} {time.time() - t0:.0f}s", flush=True)
    ids = sorted(R)
    fold = folds_for({i: {"meta": {"repo": R[i]["repo"]}} for i in ids})
    eval_ids = [i for i in ids if R[i]["n_gold"] > 0]
    oof = defaultdict(dict)
    choices = defaultdict(list)
    preds_lr, preds_gb = {}, {}
    for f in range(5):
        train = [i for i in eval_ids if fold[i] != f]
        test = [i for i in ids if fold[i] == f]
        # non-learned extensions: select config on training folds
        for fam in ("rescue", "steiner"):
            keys = [k for k in R[ids[0]]["res"] if k.startswith(fam + "|")]
            best = max(keys, key=lambda k: np.nanmean([auc(R[i]["res"][k]) for i in train]))
            choices[fam].append(best)
            for i in test:
                oof[fam][i] = R[i]["res"][best]
        for i in test:
            oof["bm25"][i] = R[i]["res"]["bm25"]
        # learned ranker
        Xtr = np.concatenate([R[i]["X"] for i in train])
        ytr = np.concatenate([R[i]["y"] for i in train])
        sc = StandardScaler().fit(Xtr)
        lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000).fit(sc.transform(Xtr), ytr)
        gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0,
                                            class_weight="balanced", random_state=0).fit(Xtr, ytr)
        choices["learned_lr_coef"].append(dict(zip(FEATS, np.round(lr.coef_[0], 3).tolist())))
        for i in test:
            preds_lr[i] = (R[i]["cand"], lr.predict_proba(sc.transform(R[i]["X"]))[:, 1])
            preds_gb[i] = (R[i]["cand"], gb.predict_proba(R[i]["X"])[:, 1])
    print(f"training done {time.time() - t0:.0f}s", flush=True)
    for name, preds in (("lr", preds_lr), ("gb", preds_gb)):
        L = score_learned(a.data, groups, preds)
        for i, r in L.items():
            oof[f"Learned ranker ({name})"][i] = r["learned"]
            oof[f"Learned ranker + tiers ({name})"][i] = r["learnedT"]
    names = {"bm25": "BM25 top-k", "rescue": "Budget-split graph rescue", "steiner": "Steiner-connected top-k (shortest paths)"}
    for k, v in names.items():
        oof[v] = oof.pop(k)
    order = ["BM25 top-k", "Budget-split graph rescue", "Steiner-connected top-k (shortest paths)",
             "Learned ranker (lr)", "Learned ranker + tiers (lr)", "Learned ranker (gb)", "Learned ranker + tiers (gb)"]
    lines = ["# Tier A extensions (out-of-fold, same 5 folds as Tier A)\n",
             f"Tasks with gold: {len(eval_ids)}. Gold inside learned candidate set: {sum(R[i]['gold_in_cand'] for i in eval_ids)}.\n",
             "| Method | " + " | ".join(f"R@{B // 1000}k" for B in BUDGETS) + " | Recall-AUC | Hit@8k | ΔAUC vs BM25 [95% CI] | Hit@8k wins/losses, p |",
             "|" + "---|" * (len(BUDGETS) + 5)]
    stats = {}
    for name in order:
        per = oof[name]
        rec = [np.nanmean([per[i][B]["recall"] for i in eval_ids]) for B in BUDGETS]
        A = np.nanmean([auc(per[i]) for i in eval_ids])
        h8 = np.nanmean([per[i][8000]["hit"] for i in eval_ids])
        if name == "BM25 top-k":
            delta, mc = "—", "—"
        else:
            d = np.array([auc(per[i]) - auc(oof["BM25 top-k"][i]) for i in eval_ids])
            m_, lo, hi = boot_ci(d)
            b = sum(per[i][8000]["hit"] == 1 and oof["BM25 top-k"][i][8000]["hit"] == 0 for i in eval_ids)
            c = sum(per[i][8000]["hit"] == 0 and oof["BM25 top-k"][i][8000]["hit"] == 1 for i in eval_ids)
            delta, mc = f"{m_:+.3f} [{lo:+.3f}, {hi:+.3f}]", f"{b}/{c}, p={mcnemar_exact(b, c):.3f}"
            stats[name] = {"auc_delta": [m_, lo, hi], "hit8": [b, c]}
        lines.append(f"| {name} | " + " | ".join(f"{x:.3f}" for x in rec) + f" | {A:.3f} | {h8:.3f} | {delta} | {mc} |")
    # subgroups for the best learned variant
    ng = {i: R[i]["n_gold"] for i in eval_ids}
    lines.append("\n## By number of gold functions (post hoc): ΔAUC vs BM25\n")
    lines.append("| Method | 1 | 2–3 | ≥4 |")
    lines.append("|---|---|---|---|")
    for name in order[1:]:
        cells = []
        for lo_, hi_ in ((1, 1), (2, 3), (4, 999)):
            s_ = [i for i in eval_ids if lo_ <= ng[i] <= hi_]
            m_, lo, hi = boot_ci(np.array([auc(oof[name][i]) - auc(oof["BM25 top-k"][i]) for i in s_]))
            cells.append(f"{m_:+.3f} [{lo:+.3f}, {hi:+.3f}] (n={len(s_)})")
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    lines.append("\n## CV choices\n")
    for k in ("rescue", "steiner"):
        lines.append(f"- {k}: " + "; ".join(choices[k]))
    lines.append("- logistic coefficients per fold (standardized features):")
    for c in choices["learned_lr_coef"]:
        lines.append("  - " + ", ".join(f"{k} {v:+.2f}" for k, v in sorted(c.items(), key=lambda kv: -abs(kv[1]))[:8]))
    open(os.path.join(a.out, "summary.md"), "w", encoding="utf-8").write("\n".join(lines) + "\n")
    with open(os.path.join(a.out, "oof_per_task.csv"), "w", encoding="utf-8") as fh:
        fh.write("instance_id,repo,fold,n_gold,method,budget,recall,recall_any,hit\n")
        for name in order:
            for i in ids:
                if i not in oof[name]:
                    continue
                for B in BUDGETS:
                    m = oof[name][i][B]
                    fh.write(f"{i},{R[i]['repo']},{fold[i]},{R[i]['n_gold']},{name},{B},{m['recall']},{m['recall_any']},{m['hit']}\n")
    json.dump({"choices": {k: v for k, v in choices.items()}, "stats": stats}, open(os.path.join(a.out, "cv_choices.json"), "w"), indent=1, default=str)
    print("\n".join(lines))
    print(f"total {time.time() - t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evtok/export_labels.py
"""Regenerate function-level localization labels for the public tasks (a Gemma 4 localization eval set).

Usage: python -m evtok.export_labels --data <DATA_DIR> --out labels.jsonl

Each line: {"instance_id", "repo", "base_commit", "gold_nodes": [dotted ids], "gold_files": [paths],
"addition_only": bool, "n_graph_nodes": int}. Labels are derived from the reference patch by textual
matching to graph nodes (see evtok.gold). Competition rules forbid redistributing competition data, so
this script is the distributable artefact: anyone with the dataset can rebuild the labels exactly.
"""
from __future__ import annotations

import argparse
import json
from collections import defaultdict

from .data import graph_key, load_graph, load_tasks
from .gold import gold_nodes


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="labels.jsonl")
    a = ap.parse_args()
    groups = defaultdict(list)
    for t in load_tasks(a.data):
        groups[graph_key(t)].append(t)
    n = 0
    with open(a.out, "w", encoding="utf-8") as f:
        for key in sorted(groups):
            g = load_graph(a.data, key)
            for t in groups[key]:
                gd = gold_nodes(g, t["patch"])
                f.write(json.dumps({
                    "instance_id": t["instance_id"], "repo": t["repo"], "base_commit": t["base_commit"],
                    "gold_nodes": sorted(g.ids[v] for v in gd["nodes"]), "gold_files": gd["files"],
                    "addition_only": gd["addition_only"], "n_graph_nodes": len(g.ids),
                }) + "\n")
                n += 1
    print(f"wrote {n} label rows to {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evtok/rl_select.py
"""RL with token penalties for context selection (REINFORCE, CPU).

MDP per task: the state is the set of functions selected so far. An action adds one candidate function
(full source) or STOPs. The reward per step is (new gold functions captured / |gold|) - lambda * tokens/1000,
so the policy trades evidence against token "fuel" and chooses its own budget per task (hard cap 16k).
Candidates and node features come from evtok.extensions (problem statement, graph and embeddings only).
The policy is a linear softmax over [node features, adjacency-to-selected, log cost] with a STOP logit
over [1, tokens used/1000, steps]. Training uses gold labels from training-fold tasks only; evaluation is
a greedy rollout on held-out tasks, compared with BM25 top-k given the SAME number of tokens per task.

Usage: python -m evtok.rl_select --data <DATA_DIR> --out results/rl --workers 8
"""
from __future__ import annotations

import argparse
import json
import math
import os
import time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

from . import selectors as S
from .analyze import boot_ci, folds_for, mcnemar_exact
from .data import graph_key, load_graph, load_tasks
from .extensions import FEATS, task_features
from .gold import gold_nodes
from .retrieval import task_signals

CAP = 16000
LAMBDAS = (0.02, 0.05, 0.1)  # penalty per 1,000 tokens
MAX_STEPS = 60


def prep_group(data_dir, key, tasks):
    g = load_graph(data_dir, key)
    out = {}
    for t in tasks:
        gold = gold_nodes(g, t["patch"])
        sig = task_signals(g, t["problem_statement"])
        sig["_problem"] = t["problem_statement"]
        cand, X, order, _ = task_features(g, sig)
        pos = {int(v): i for i, v in enumerate(cand)}
        adj = [[pos[int(n)] for n in g.nbrs[int(v)] if int(n) in pos] for v in cand]
        bm_order = [int(v) for v in order[:3000]]
        out[t["instance_id"]] = {
            "repo": t["repo"], "n_gold": len(gold["nodes"]), "cand": cand, "X": X,
            "y": np.array([int(v) in gold["nodes"] for v in cand], dtype=np.int8),
            "cost": g.cost[cand, 2].astype(np.float64), "adj": adj,
            "bm_order": bm_order, "bm_cost": {v: int(g.cost[v, 2]) for v in bm_order},
            "gold": sorted(gold["nodes"]),
        }
    return out


class Policy:
    def __init__(self, d, rng):
        self.w = rng.normal(0, 0.01, d)
        self.ws = np.zeros(3)
        self.ws[0] = -2.0  # start biased against stopping

    def logits(self, F, stop_feat):
        return np.append(F @ self.w, stop_feat @ self.ws)


def node_feats(T, mu, sd, selected_mask, adj_count):
    Xs = (T["X"] - mu) / sd
    return np.column_stack([Xs, np.minimum(adj_count, 3) / 3.0, np.log1p(T["cost"]) / 8.0])


def rollout(T, pol, mu, sd, lam, rng, greedy=False):
    n = len(T["cand"])
    sel = np.zeros(n, bool)
    adj_count = np.zeros(n)
    used, steps = 0.0, 0
    traj = []  # (F, stop_feat, valid, action, reward)
    ng = max(1, int(T["y"].sum())) if T["y"].sum() else max(1, T["n_gold"])
    base = node_feats(T, mu, sd, sel, adj_count)
    while steps < MAX_STEPS:
        valid = (~sel) & (used + T["cost"] <= CAP)
        stop_feat = np.array([1.0, used / 1000.0, steps / 10.0])
        F = base.copy()
        F[:, -2] = np.minimum(adj_count, 3) / 3.0
        lg = pol.logits(F, stop_feat)
        lg[:-1][~valid] = -1e9
        lg = lg - lg.max()
        p = np.exp(lg)
        p /= p.sum()
        a = int(np.argmax(p)) if greedy else int(rng.choice(len(p), p=p))
        if a == n or not valid.any():
            traj.append((F, stop_feat, valid, n, 0.0, p))
            break
        r = T["y"][a] / ng - lam * T["cost"][a] / 1000.0
        traj.append((F, stop_feat, valid, a, r, p))
        sel[a] = True
        used += T["cost"][a]
        for j in T["adj"][a]:
            adj_count[j] += 1
        steps += 1
    return traj, sel, used


def reinforce(train, R, mu, sd, lam, seed, epochs=40, samples=4, lr=0.05):
    rng = np.random.default_rng(seed)
    d = len(FEATS) + 2
    pol = Policy(d, rng)
    baseline = defaultdict(float)
    for ep in range(epochs):
        order = rng.permutation(train)
        for iid in order:
            T = R[iid]
            gw, gs = np.zeros_like(pol.w), np.zeros_like(pol.ws)
            for _ in range(samples):
                traj, _, _ = rollout(T, pol, mu, sd, lam, rng)
                rets = np.cumsum([x[4] for x in traj][::-1])[::-1]
                G0 = rets[0] if len(rets) else 0.0
                for (F, sf, valid, a, r, p), Gt in zip(traj, rets):
                    adv = Gt - baseline[iid]
                    n = F.shape[0]
                    pn = p[:n] * valid
                    if a == n:
                        gs += adv * sf * (1 - p[n])
                        gw -= adv * (pn @ F)
                    else:
                        gw += adv * (F[a] - pn @ F)
                        gs -= adv * p[n] * sf
                baseline[iid] = 0.9 * baseline[iid] + 0.1 * G0
            pol.w += lr * gw / samples
            pol.ws += lr * gs / samples
    return pol


def bm25_at(T, tokens):
    pk, used = [], 0
    for v in T["bm_order"]:
        c = T["bm_cost"][v]
        if used + c <= tokens:
            pk.append(v)
            used += c
    gold = set(T["gold"])
    hit = len(gold & set(pk))
    return hit / len(gold), float(hit > 0), used


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results/rl")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--seeds", type=int, default=3, help="independent training seeds; seed 0 is the headline run")
    a = ap.parse_args()
    import sys
    sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    t0 = time.time()
    groups = defaultdict(list)
    for t in load_tasks(a.data):
        groups[graph_key(t)].append(t)
    R = {}
    with ProcessPoolExecutor(max_workers=a.workers) as ex:
        for f in as_completed([ex.submit(prep_group, a.data, k, v) for k, v in sorted(groups.items())]):
            R.update(f.result())
    print(f"prepared {len(R)} tasks {time.time() - t0:.0f}s", flush=True)
    ids = sorted(R)
    fold = folds_for({i: {"meta": {"repo": R[i]["repo"]}} for i in ids})
    eval_ids = [i for i in ids if R[i]["n_gold"] > 0]
    jobs = [(lam, f, s) for s in range(a.seeds) for lam in LAMBDAS for f in range(5)]
    by_seed = defaultdict(dict)  # (lam, seed) -> iid -> dict
    with ProcessPoolExecutor(max_workers=a.workers) as ex:
        futs = {ex.submit(train_eval, R, fold, eval_ids, lam, f, a.epochs, s): (lam, f, s) for lam, f, s in jobs}
        for n, fu in enumerate(as_completed(futs), 1):
            lam, f, s = futs[fu]
            by_seed[(lam, s)].update(fu.result())
            print(f"[{n}/{len(jobs)}] seed {s} lambda {lam} fold {f} done {time.time() - t0:.0f}s", flush=True)
    results = {lam: by_seed[(lam, 0)] for lam in LAMBDAS}  # headline table: seed 0
    lines = ["# RL with token penalties for context selection (REINFORCE, out-of-fold)\n",
             f"Tasks with gold: {len(eval_ids)}. Reward per step = gold captured / |gold| − λ · tokens/1000; cap {CAP} tokens. "
             "Baseline: BM25 top-k given exactly the tokens the policy spent on each task.\n",
             "| λ per 1k tokens | mean tokens used | RL recall | BM25 recall at same tokens | Δ recall [95% CI] | RL Hit | BM25 Hit | Hit wins/losses, p | mean stop step |",
             "|---|---|---|---|---|---|---|---|---|"]
    summary = {}
    for lam in LAMBDAS:
        rr = results[lam]
        rec = np.array([rr[i]["rl_recall"] for i in eval_ids])
        brec = np.array([rr[i]["bm_recall"] for i in eval_ids])
        m_, lo, hi = boot_ci(rec - brec)
        b = sum(rr[i]["rl_hit"] == 1 and rr[i]["bm_hit"] == 0 for i in eval_ids)
        c = sum(rr[i]["rl_hit"] == 0 and rr[i]["bm_hit"] == 1 for i in eval_ids)
        tok = np.mean([rr[i]["tokens"] for i in eval_ids])
        lines.append(f"| {lam} | {tok:.0f} | {rec.mean():.3f} | {brec.mean():.3f} | {m_:+.3f} [{lo:+.3f}, {hi:+.3f}] | "
                     f"{np.mean([rr[i]['rl_hit'] for i in eval_ids]):.3f} | {np.mean([rr[i]['bm_hit'] for i in eval_ids]):.3f} | "
                     f"{b}/{c}, p={mcnemar_exact(b, c):.3f} | {np.mean([rr[i]['steps'] for i in eval_ids]):.1f} |")
        toks = np.array([rr[i]["tokens"] for i in eval_ids])
        summary[lam] = {"tokens_mean": tok, "tokens_p10_p90": [float(np.percentile(toks, 10)), float(np.percentile(toks, 90))],
                        "rl_recall": float(rec.mean()), "bm_recall": float(brec.mean()), "delta": [m_, lo, hi], "hit_wins_losses": [b, c]}
    lines.append("\nToken spend varies per task (10th–90th percentile): " +
                 "; ".join(f"λ={l}: {summary[l]['tokens_p10_p90'][0]:.0f}–{summary[l]['tokens_p10_p90'][1]:.0f}" for l in LAMBDAS))
    if a.seeds > 1:
        lines.append(f"\n## Stability across {a.seeds} training seeds\n")
        lines.append("The table above is seed 0. Each seed retrains every fold from a different initialization and sampling stream.\n")
        lines.append("| λ per 1k tokens | mean tokens, per seed | Δ recall vs BM25, per seed | mean Δ | seeds with RL below BM25 |")
        lines.append("|---|---|---|---|---|")
        below = total = 0
        for lam in LAMBDAS:
            toks, deltas = [], []
            for s in range(a.seeds):
                rr = by_seed[(lam, s)]
                toks.append(np.mean([rr[i]["tokens"] for i in eval_ids]))
                deltas.append(np.mean([rr[i]["rl_recall"] - rr[i]["bm_recall"] for i in eval_ids]))
            below += sum(d < 0 for d in deltas)
            total += len(deltas)
            lines.append(f"| {lam} | " + ", ".join(f"{x:.0f}" for x in toks) + " | " + ", ".join(f"{d:+.3f}" for d in deltas) +
                         f" | {np.mean(deltas):+.3f} | {sum(d < 0 for d in deltas)}/{len(deltas)} |")
            summary[lam]["per_seed"] = {"tokens": [float(x) for x in toks], "delta": [float(d) for d in deltas]}
        lines.append(f"\nRL is below BM25 at matched tokens in {below} of {total} (λ, seed) runs.")
    open(os.path.join(a.out, "summary.md"), "w", encoding="utf-8").write("\n".join(lines) + "\n")
    json.dump(summary, open(os.path.join(a.out, "summary.json"), "w"), indent=1)
    print("\n".join(lines))
    print(f"total {time.time() - t0:.0f}s")


def train_eval(R, fold, eval_ids, lam, f, epochs, s=0):
    train = [i for i in eval_ids if fold[i] != f]
    test = [i for i in eval_ids if fold[i] == f]
    Xtr = np.concatenate([R[i]["X"] for i in train])
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    pol = reinforce(train, R, mu, sd, lam, seed=f + 1000 * s, epochs=epochs)  # s=0 reproduces the single-seed run
    out = {}
    rng = np.random.default_rng(0)
    for i in test:
        T = R[i]
        traj, sel, used = rollout(T, pol, mu, sd, lam, rng, greedy=True)
        gold = set(T["gold"])
        chosen = set(int(v) for v in T["cand"][sel])
        rl_rec = len(gold & chosen) / len(gold)
        bm_rec, bm_hit, _ = bm25_at(T, used)
        out[i] = {"rl_recall": rl_rec, "rl_hit": float(rl_rec > 0), "bm_recall": bm_rec, "bm_hit": bm_hit,
                  "tokens": used, "steps": int(sel.sum())}
    return out


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tests/test_core.py
"""Minimal unit tests: budget enforcement, connectivity, patch parsing, tokenization."""
import numpy as np

from evtok import selectors as S
from evtok.data import RepoGraph
from evtok.gold import parse_patch, module_prefixes
from evtok.textproc import ident_tokens, signature_text


def toy_graph():
    ids = ["pkg.a", "pkg.b", "pkg.c", "pkg.d", "tests.test_x"]
    text = ["def a():\n    return b()", "def b():\n    return c()", "def c():\n    pass", "def d():\n    pass", "def test_x():\n    a()"]
    nbrs = [np.array([1, 4]), np.array([0, 2]), np.array([1]), np.array([], dtype=np.int64), np.array([0])]
    cost = np.array([[3, 6, 20], [3, 6, 20], [3, 6, 20], [3, 6, 20], [3, 6, 20]])
    pool = np.array([True, True, True, True, False])
    return RepoGraph("toy", ids, text, {k: i for i, k in enumerate(ids)}, nbrs, np.zeros((5, 4)), np.zeros(5, bool), cost, pool)


def test_budget_never_exceeded():
    g = toy_graph()
    s = np.array([3.0, 2.0, 1.0, 2.5, -1e9])
    for B in (5, 10, 25, 47, 100):
        for expand in (False, True):
            pk = S.greedy_tiers(g, S.utility(s, None, 0, 1, 1), B, [0, 3] if not expand else [0], expand=expand)
            assert S.packet_cost(g, pk) <= B
        assert S.packet_cost(g, S.flat_full(g, np.argsort(-s), B)) <= B


def test_connected_only_reaches_neighbours_and_skips_tests():
    g = toy_graph()
    s = np.array([3.0, 2.0, 1.0, 2.5, 5.0])
    pk = S.greedy_tiers(g, S.utility(s, None, 0, 1, 1), 1000, [0], expand=True)
    assert 3 not in pk  # disconnected from seed
    assert 4 not in pk  # test node excluded from pool
    assert set(pk) == {0, 1, 2}


def test_parse_patch_and_prefixes():
    p = "--- a/pkg/mod.py\n+++ b/pkg/mod.py\n@@ -1,2 +1,2 @@ def f():\n-    x = 1\n+    x = 2\n"
    f = parse_patch(p)
    assert f[0]["path"] == "pkg/mod.py" and f[0]["hunks"][0]["lines"][0] == ("-", "    x = 1")
    assert module_prefixes("docs_src/a/b.py") == ["docs_src.a.b", "a.b"]
    assert module_prefixes("rich/__init__.py") == ["rich"]


def test_tokens_and_signature():
    assert "parse" in ident_tokens("parseHeader") and "parse_header" in ident_tokens("parse_header")
    sig = signature_text('def f(a,\n      b):\n    """Doc."""\n    return a')
    assert sig.startswith("def f(a,") and '"""Doc."""' in sig and "return" not in sig

## 2. Unit tests

Budget bound, connectivity, test-node exclusion, patch parsing, tokenization.

In [ ]:
run("-m", "pytest", "-q", "tests")

## 3. Evaluate every configuration on every task

Writes `results/tier_a/grid.pkl`. No gold information is used to build packets.

In [ ]:
run("-m", "evtok.run_tier_a", "--data", DATA, "--out", "results/tier_a", "--workers", str(WORKERS))

## 4. Cross-validated selection and statistics

Writes `summary.md`, `summary.csv`, `oof_per_task.csv` and `cv_choices.json`.

In [ ]:
run("-m", "evtok.analyze", "--grid", "results/tier_a/grid.pkl", "--out", "results/tier_a")
from IPython.display import Markdown, display
display(Markdown(open("results/tier_a/summary.md", encoding="utf-8").read()))

## 5. Structural headroom diagnostics

How far are gold functions from what BM25 retrieved? Writes `diagnostics.csv`.

In [ ]:
run("-m", "evtok.diagnostics", "--data", DATA, "--out", "results/tier_a")

## 6. Subgroup analysis by number of gold functions (exploratory, post hoc)

In [ ]:
import csv, collections
import numpy as np
rows = list(csv.DictReader(open("results/tier_a/oof_per_task.csv", encoding="utf-8")))
d = collections.defaultdict(lambda: collections.defaultdict(list)); ng = {}
for r in rows:
    if r["n_gold"] == "0":
        continue
    d[r["method"]][r["instance_id"]].append(float(r["recall"])); ng[r["instance_id"]] = int(r["n_gold"])
def ci(x):
    x = np.array(x); rng = np.random.default_rng(0)
    m = x[rng.integers(0, len(x), (10000, len(x)))].mean(1)
    return f"{x.mean():+.3f} [{np.percentile(m, 2.5):+.3f}, {np.percentile(m, 97.5):+.3f}]"
for lo, hi in ((1, 1), (2, 3), (4, 999)):
    s = [i for i in ng if lo <= ng[i] <= hi]
    for m in ("Connected + tiers (EpT)", "Hybrid + tiers (no graph)", "Graph expansion (full only)"):
        print(f"gold {lo}-{hi if hi < 999 else '+'} (n={len(s)})  {m} - BM25: {ci([np.mean(d[m][i]) - np.mean(d['BM25 top-k'][i]) for i in s])}")

## 7. Figure 1

In [ ]:
run("-m", "evtok.figures", "--csv", "results/tier_a/oof_per_task.csv", "--out", "results/tier_a")
from IPython.display import Image
Image("results/tier_a/fig_recall_vs_budget.png")

## 8. Robustness: learned ranker, budget-split rescue, Steiner-connected top-k (paper Section 6.3)

Same 5 folds. The ranker (logistic regression and gradient-boosted trees, scikit-learn) is trained per fold on training tasks only and scored on held-out tasks.

In [ ]:
from IPython.display import Markdown, display
run("-m", "evtok.extensions", "--data", DATA, "--out", "results/tier_a_ext", "--workers", str(WORKERS))
display(Markdown(open("results/tier_a_ext/summary.md", encoding="utf-8").read()))

## 9. RL with token penalties for context selection (paper Section 6.4)

REINFORCE policy that adds functions one at a time or stops. Reward = gold captured / |gold| − λ · tokens/1000, with λ ∈ {0.02, 0.05, 0.1}. Trained per fold on training tasks only. Each held-out task is compared with BM25 given the same number of tokens the policy chose to spend. Training is repeated with three seeds to show how stable the result is; the first table is seed 0. About 30 minutes on 4 CPU cores.

In [ ]:
from IPython.display import Markdown, display
run("-m", "evtok.rl_select", "--data", DATA, "--out", "results/rl", "--workers", str(WORKERS), "--epochs", "40", "--seeds", "3")
display(Markdown(open("results/rl/summary.md", encoding="utf-8").read()))

## 10. Localization labels (evaluation set)

Regenerates gold function labels from the attached dataset. The file is deleted afterwards because competition data may not be redistributed; run this cell yourself to get it.

In [ ]:
run("-m", "evtok.export_labels", "--data", DATA, "--out", "/tmp/labels.jsonl")
import json
rows = [json.loads(l) for l in open("/tmp/labels.jsonl", encoding="utf-8")]
print(sum(not x["addition_only"] for x in rows), "of", len(rows), "tasks have function-level labels")
os.remove("/tmp/labels.jsonl")

## Notes

- Token costs are `len(text) / 3.6`, an estimate shared by all methods. Absolute budgets are approximate.
- 9 of 129 tasks only add new code and have no gold function nodes. They are excluded from function-level metrics.
- The last cell deletes `grid.pkl`. It is large and contains gold node ids derived from the reference patches. The CSV and Markdown outputs are enough to check every number in the paper.

In [ ]:
os.remove("results/tier_a/grid.pkl")
print(sorted(os.listdir("results/tier_a")))

## Extended research log

This log records how the study was designed, what we decided at each step, and why. It goes beyond the 3,000-word Writeup. Every number in it comes from this notebook's outputs or from the dataset.

### Starting ideas

We began with three design ideas for local coding agents, first stated as physical metaphors and then rewritten as testable methods.

| Original idea | Technical form | What happened |
|---|---|---|
| Graph "gravity" with a sluice gate: the issue pulls on relevant code, and context flow stops at a hard token line | Score nodes by relevance and graph proximity; select a connected subgraph under a hard token budget, greedily by value per token | Implemented and tested (connected and connected + tiers selectors). Connectivity costs 8.6 recall points. |
| Elastic AST lattice: "pluck" the failing node and follow the vibration through dependencies | Propagate a failure signal from failing tests or stack frames along call paths (static test-path proximity) | Designed, then ruled out by the data audit: no fail-to-pass lists, empty hints, one traceback in 129 tasks. `test_patch` is the only test signal, and using it would leak the answer. |
| Staged trajectories: RL that penalizes tokens, tool calls, graph hops and retries | Reward = captured gold − λ·tokens on the context-selection MDP (REINFORCE) | **Tested on selection (section 9).** The penalty cuts mean spend from 13.0k tokens (λ = 0.02) to 2.8k and 3.0k (λ = 0.05 and 0.1), chosen per task. But the policy ranks worse than BM25 given the same tokens (Δ −0.073 to −0.116 in this notebook). RL training is sensitive to the platform: the same code on our machine gave Δ −0.067 to −0.111, with the same conclusion. Section 9 therefore repeats training with three seeds and reports each one. RL fine-tuning of Gemma over tool calls is still deferred: 129 tasks, costly containerized rollouts, and the risk of agents gaming tests. |

We kept one contribution: a deterministic, budgeted context selector. We dropped claims of optimality and of matching cloud models, since nothing here could support them.

### Asset audit (what the data actually contains)

- 129 tasks: fastapi 67, rich 48, requests 13, httpx 1. Problem statements are PR descriptions (median 418 characters), many of them features or refactors.
- Graphs carry only node id, name and source text, with no file paths or line numbers. There is one edge type (`calls`), and 23% of non-test nodes have no edges.
- Embeddings are 256-d with full coverage. The encoder is not provided, and `search_similar_code` only accepts symbol names. So we built dense queries from identifier anchors or from BM25 seeds.
- **Leakage check:** 77.5% of the lines each fix deletes appear in the graph text, against 4.6% of the lines it adds. The graphs reflect the base commit.
- **Gold mapping:** we match the reference patch's removed and context lines against node text. This labels 120 tasks; 9 only add code.

### Pre-registered decision gates

| Gate | Condition | Outcome |
|---|---|---|
| G0 leakage | No gold-derived input to selection; graphs pre-fix | Passed |
| G1 retrieval | EpT beats hybrid top-k at 3 or more budgets, with the CI excluding 0 | Failed. ΔAUC −0.028 [−0.069, +0.012]. Pivoted, as planned, to "when does structure help". |
| G2 harness | End-to-end tests runnable | The `swegemma` harness ships in the Kaggle wheelhouse but not in the dataset zip. No end-to-end repair was run. |

### Alternatives considered

| Alternative | Decision | Reason |
|---|---|---|
| Pure heuristic packing | Main method | Deterministic and CPU-only |
| Multi-level (semantic) compression | Integrated as tiers | Recovers most of the connectivity cost; +9.2 on multi-function edits |
| Hybrid BM25 + dense | Baseline and seed retrieval | Per-fold tuning overfit slightly; untuned BM25 is best |
| Learned node / edit-location ranker | Tested (section 8 of this notebook) | Logistic regression and boosted trees on 18 features, trained per fold on training tasks only. Neither beats BM25 (ΔAUC −0.025 and −0.055). With tiers, logistic regression gains +0.110 on edits of four or more functions, replicating the tier effect. |
| Iterative refinement | Tested as a static budget split | 25–50% of the budget goes to neighbours of the top hits: −0.012 vs BM25. A model-triggered version needs an agent loop. |
| Prize-collecting Steiner tree (G-Retriever) | Tested as an approximation | Seeds connected by shortest paths: −0.002 vs BM25. Exact PCST would need `pcst_fast`, which is GPL-licensed, so it was not shipped. |
| Graph serialization ablation (Talk like a Graph) | Built (GPU notebook) | The same EpT packet rendered with no graph text, with a call-edge list, or with per-entry adjacency lines |
| Gemma 4 localization probe + loss waterfall | Built (GPU notebook) | Tests whether the model uses evidence that is present |
| Selector as an ADK skill script | Built | A stdlib BM25 localizer packaged as a skill in a main-track submission variant. It ranked a gold file first on 3 of 3 real snapshots tested offline. Not yet scored by the harness. |
| Localization labels as an evaluation set | Built (`evtok.export_labels`) | Regenerates gold function labels for 120 tasks from the dataset. The output is not redistributed. |

### Competition rules that shaped the release

Competition data may not be redistributed, so this notebook regenerates everything from the attached dataset. Only derived metrics are kept as outputs. Code is public on Kaggle under Apache-2.0. Dependencies are limited to numpy, matplotlib, scikit-learn and pytest, all preinstalled on Kaggle.

### Threats to validity

- Token costs are estimates (characters / 3.6).
- Gold labels come from text matching.
- Four Python libraries, mostly fastapi.
- The subgroup analysis is post hoc.
- No resolve-rate measurement.